In [81]:
import pandas as pd
import numpy as np

In [82]:
df = pd.read_csv("01 covid_toy.csv")
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [83]:
print(df.cough.value_counts())
print()
print()
print(df.city.value_counts())


cough
Mild      62
Strong    38
Name: count, dtype: int64


city
Kolkata      32
Bangalore    30
Delhi        22
Mumbai       16
Name: count, dtype: int64


In [84]:
'''
age - numeric column
gender - nominal categorical column(one hot encoding)
fever - numeric column.has missing values.will apply simple imputer
cough - nominal categorical column(ordinal encoding)
city - nominal categorical column(one hot encoding)
has_covid - nominal categorical column(label encoding)

here we will work on nominal categorical columns
'''


'\nage - numeric column\ngender - nominal categorical column(one hot encoding)\nfever - numeric column.has missing values.will apply simple imputer\ncough - nominal categorical column(ordinal encoding)\ncity - nominal categorical column(one hot encoding)\nhas_covid - nominal categorical column(label encoding)\n\nhere we will work on nominal categorical columns\n'

In [85]:
df.isnull().sum()

,0
age,0
gender,0
fever,10
cough,0
city,0
has_covid,0


In [86]:
x = df.iloc[:,0:5]
y = df.drop(x,axis='columns')

print(x.head())

print()
print()
print()

print(y.head())

   age  gender  fever cough     city
0   60    Male  103.0  Mild  Kolkata
1   27    Male  100.0  Mild    Delhi
2   42    Male  101.0  Mild    Delhi
3   31  Female   98.0  Mild  Kolkata
4   65  Female  101.0  Mild   Mumbai



  has_covid
0        No
1       Yes
2        No
3        No
4        No


In [104]:
#train-test split

from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2)
print(x_train.head())
print()
print()
print(y.head())

    age  gender  fever   cough       city
28   16    Male  104.0    Mild    Kolkata
58   23    Male   98.0  Strong     Mumbai
1    27    Male  100.0    Mild      Delhi
69   73  Female  103.0    Mild      Delhi
18   64  Female   98.0    Mild  Bangalore


  has_covid
0        No
1       Yes
2        No
3        No
4        No


In [88]:
#without column transformer

#performing simpleImputer on fever column for filing missing values

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder

In [89]:
si = SimpleImputer()
le = LabelEncoder()

In [90]:
#applying label encoding on class column

y_train.has_covid = le.fit_transform(y_train.has_covid)
y_test.has_covid = le.transform(y_test['has_covid'])

In [91]:
x_train['fever']=si.fit_transform(x_train[['fever']])
x_test['fever']=si.transform(x_test[['fever']])

In [92]:
from sklearn.preprocessing import OrdinalEncoder

In [93]:
oe = OrdinalEncoder()

x_train['cough'] = oe.fit_transform(x_train[['cough']])
x_test['cough'] = oe.transform(x_test[['cough']])

In [94]:
from sklearn.preprocessing import OneHotEncoder

In [95]:
ohe = OneHotEncoder(drop ='first',sparse_output = False)

In [96]:
x_train_gender_city = ohe.fit_transform(x_train[['gender','city']])
x_test_gender_city = ohe.transform(x_test[['gender','city']])

In [97]:
print(x_train_gender_city.shape)
print(x_test_gender_city.shape)

(80, 4)
(20, 4)


In [98]:
x_train_gender_city[1]

array([0., 0., 1., 0.])

In [99]:
#extracting values of age

x_train_age=x_train['age'].values
x_test_age=x_test['age']

In [100]:
#column transformer

from sklearn.compose import ColumnTransformer

In [101]:
transformer = [('trans1',SimpleImputer(),['fever']),
              ('trans2',OrdinalEncoder(categories=[['Mild','Strong']]),['cough']),
              ('trans3',OneHotEncoder(sparse_output = False,drop = 'first'),['gender','city'])]

In [102]:
ct = ColumnTransformer(transformers=transformer,remainder='passthrough').set_output(transform='pandas')

In [105]:
x_train_trans = ct.fit_transform(x_train)

In [107]:
x_train_trans.sample(5)

,trans1__fever,trans2__cough,trans3__gender_Male,trans3__city_Delhi,trans3__city_Kolkata,trans3__city_Mumbai,remainder__age
92,102.0,1.0,0.0,0.0,1.0,0.0,82
18,98.0,0.0,0.0,0.0,0.0,0.0,64
96,101.0,1.0,0.0,0.0,1.0,0.0,51
67,99.0,0.0,1.0,0.0,0.0,0.0,65
85,103.0,0.0,0.0,0.0,0.0,0.0,16
